In [1]:
# NEEDED FOR RESAMPLING USING TORCHAUDIO
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# import torch

# torch.set_num_threads(1)
# torch.set_num_interop_threads(1)

In [2]:
import sys
from tqdm import tqdm

BASE_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr"
VOCAB_FILE = f"{BASE_PATH}/src/model/xeusphoneme/resources/ipa_vocab.json"

sys.path.append(BASE_PATH)

# ipapack
from src.data.kaldi_pretraining_dataset import build_kaldi_datamodule

datamodule = build_kaldi_datamodule(
    train_split="dev_1k",  # "train_accentmix_multi",
    dev_splits=[
        # "dev_1k",
        "dev_gmuaccent",
        # "dev_buckeye",
        # "dev_epadb",
        # "dev_speechoceanotth",
        # "dev_l2arctic",
    ],
    predict_split="predict",
    dataset_config_path=f"{BASE_PATH}/configs/data/ipapack_index.yaml",
    batch_size=1,
    num_workers=1,
    vocab_file=VOCAB_FILE,
    # limit_samples=2,
)

# from src.data.kaldi_dataset import build_kaldi_datamodule

# DATASET = "buckeye"
# datamodule = build_kaldi_datamodule(
#     DATASET,
#     data_dir="/work/hdd/bbjs/shared/powsm/s2t1/dump/raw",
#     dataset_config_path=f"{BASE_PATH}/configs/data/powsm_evalset_index.yaml",
#     portable_wavscp=False,
#     sampling_rate=16000,
#     batch_size=1,
#     num_workers=1,
#     vocab_file=VOCAB_FILE,
# )

datamodule.setup()
dataloader = datamodule.val_dataloader()
dataloader = iter(dataloader)

print("Loaded dataset with length:", len(dataloader))

/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/.venv_dai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 4655 samples from wav.scp: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/dev_1k_fixed/wav.scp with count 0


Reading text: 18620it [00:00, 110226.87it/s]
Reading language: 18620it [00:00, 107459.39it/s]


Loaded 318 samples from wav.scp: /work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/data/devsets/gmuaccent.scp with count 0


Reading text: 1242it [00:00, 6433.09it/s]
Reading language: 1242it [00:00, 143804.71it/s]


Loaded 4655 samples from wav.scp: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/dev_1k_fixed/wav.scp with count 0


Reading text: 18620it [00:00, 621843.45it/s]
Reading language: 18620it [00:00, 602217.24it/s]


Loaded dataset with length: 318


In [19]:
import torch
from src.recipe.phone_recognition.model_module import PhoneRecognitionModel

# from src.model.xeusphoneme.builders import build_xeus_pr_from_hf
# from src.recipe.phone_recognition.greedy_ctc_strategy import GreedyCTCInference
from src.model.xeusphoneme.builders import build_xeus_pr_inference
import json

BASE_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr"
CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/ipaaccent_ctc/xeus_multiaccent.panphon.bs256.lr3em5.sched_p15warm_p85const_3kunfreeze.40ksteps/checkpoints/checkpoint-12000.ckpt"

# CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/ipaaccent_ctc/xeus_multiaccent.bs256.lr3em5.sched_p15warm_p85const_3kunfreeze.40ksteps/checkpoints/checkpoint-12000.ckpt"
VOCAB_FILE = f"{BASE_PATH}/src/model/xeusphoneme/resources/ipa_vocab.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device = ", device)


work_dir = f"{BASE_PATH}/exp/cache/xeus"
device = "cpu" if not torch.cuda.is_available() else "cuda:0"
inference = build_xeus_pr_inference(
    work_dir=work_dir,
    checkpoint=CKPT_PATH,
    vocab_file=VOCAB_FILE,
    hf_repo="espnet/xeus",
    config_file=None,
    device=device,
    force_download=False,
)

# net = build_xeus_pr_from_hf(
#     work_dir=f"{BASE_PATH}/exp/cache/xeus",
#     checkpoint=CKPT_PATH,
#     vocab_file=VOCAB_FILE,
#     config_file=None,
#     hf_repo="espnet/xeus",
# )

# model = PhoneRecognitionModel(net=net, optimizer=None)
# model.set_inference_strategy(GreedyCTCInference)

# model.to(device)
# model.eval()

with open(VOCAB_FILE, "r") as f:
    vocab = json.load(f)
id2token = {k: v for v, k in vocab.items()}

Returning existing local_dir `/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/cache/xeus` as remote repo cannot be accessed in `snapshot_download` (None).


Using device =  cuda


In [20]:
# import torch
# from src.model.wav2vec2.builders import build_wav2vec2pr_inference
# from src.model.xeusphoneme.builders import build_xeus_pr_from_hf
# from src.recipe.phone_recognition.model_module import PhoneRecognitionModel
# import json

# CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/ipaaccent_ctc/xeus_multiaccent.panphon.bs256.lr3em5.sched_p15warm_p85const_3kunfreeze.40ksteps/checkpoints/checkpoint-12000.ckpt"
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Using device = ", device)

# inference = build_wav2vec2pr_inference(
#     hf_repo="facebook/mms-300m",
#     vocab_file=VOCAB_FILE,
#     checkpoint=CKPT_PATH,
# )

# with open(VOCAB_FILE, "r") as f:
#     vocab = json.load(f)
# id2token = {k: v for v, k in vocab.items()}

In [21]:
from src.metrics.phone_recognition import PhoneRecognitionEvaluator

evaluator = PhoneRecognitionEvaluator()


def get_phone_str(token_ids):
    return "/".join([id2token[t] for t in token_ids if t in id2token])


N_SAMPLES = 150
n_data = len(dataloader)
print("Total data samples:", n_data, "sampling ", N_SAMPLES)
results = []
with torch.no_grad():
    for bidx, batch in tqdm(enumerate(dataloader), desc="Making predictions"):
        # if bidx % (n_data // min(N_SAMPLES, n_data)) != 0:
        #     continue
        # assert batch size is 1
        # print(batch)
        batch, _, _ = batch
        batch = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

        prediction = inference(**batch)
        for i, key in enumerate(batch["keys"]):
            pred = prediction[i]["processed_transcript"]
            gt_str = batch["text"][i]
            if isinstance(gt_str, torch.Tensor):
                gt_str = gt_str.cpu().numpy()
                gt_phones = get_phone_str(gt_str)
            else:
                gt_phones = gt_str
            prmetrics, _ = evaluator.evaluate(
                {i: {"prediction": pred, "transcription": gt_phones.replace("/", "")}},
                compute_inventory=False,
            )
            asr_text = batch["asr_text"][i] if "asr_text" in batch else ""
            results.append(
                {
                    "key": key,
                    "speech": batch["speech"][i].cpu().numpy(),
                    "speech_length": batch["speech_length"][i].cpu().item(),
                    "wavpath": batch["wavpath"][i],
                    "phone_str": gt_phones,
                    "language": batch["lang_sym"][i],
                    "asr_text": asr_text,
                    "prediction": prediction[i]["predicted_transcript"],
                    "pr_metrics": prmetrics,
                }
            )
        if bidx > 1:
            break
print(f"Collected {len(results)} samples for error analysis.")

Total data samples: 318 sampling  150


Making predictions: 0it [00:00, ?it/s]WARNING:root:Flash Attention failed, falling back to default attention: FlashAttention only supports fp16, bf16, and fp8_e4m3 data type
Making predictions: 2it [00:04,  2.23s/it]

Collected 3 samples for error analysis.


In [22]:
def play_audio(sp):
    import IPython.display as ipd

    return ipd.Audio(sp, rate=16000)

In [ ]:
print("IPA:", "ð ʃ ɲ ʔ ɑː t͡ʃ d͡ʒ")

UTF-8
IPA: ð ʃ ɲ ʔ ɑː t͡ʃ d͡ʒ


In [ ]:
from IPython.display import HTML, display
import html

FONTS = '"Noto Sans IPA","Doulos SIL","Charis SIL","DejaVu Sans",sans-serif'
FIELDS = [
    ("WAVPATH", "wavpath"),
    ("Language", "language"),
    ("ASR Text", "asr_text"),
    ("Ground Truth Phones", "phone_str"),
    ("Predicted Phones", "prediction"),
    ("PR Metrics", "pr_metrics"),
]

STYLE = f"""
<style>
.card{{font-family:{FONTS};font-size:14px;line-height:1.35;border:1px solid rgba(128,128,128,.35);
border-radius:8px;padding:10px 12px;margin:10px 0 6px;white-space:pre-wrap;font-variant-ligatures:none}}
.row{{display:grid;grid-template-columns:170px 1fr;gap:10px;margin:2px 0}}
.k{{font-weight:700;opacity:.9}} .v{{word-break:break-word}}
</style>
"""


def show_res(res):
    rows = "\n".join(
        f"<div class='row'><span class='k'>{html.escape(k)}:</span>"
        f"<span class='v'>{html.escape('' if res.get(f) is None else str(res.get(f)))}</span></div>"
        for k, f in FIELDS
    )
    display(HTML(STYLE + f"<div class='card'>{rows}</div>"))

In [28]:
for res in results:
    show_res(res)
    display(play_audio(res["speech"]))
    display(HTML("<hr style='border:none;border-top:1px solid rgba(128,128,128,.35)'>"))

In [24]:
for res in results:
    print("WAVPATH:", res["wavpath"])
    print("Language:", res["language"])
    print("ASR Text:", res["asr_text"])
    print("Ground Truth Phones:", res["phone_str"])
    print("Predicted Phones:", res["prediction"])
    print("PR Metrics:", res["pr_metrics"])
    display(play_audio(res["speech"]))
    print("-" * 40)

WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/romanian6.wav
Language: eng
ASR Text: None
Ground Truth Phones: p/l/i/z/k/ɔ/l/s/t/ɛ/l/ə/æ/s/k/h/t/u/b/r/ɪ̃/ŋ/k/d̪/i/s/t̪/ɪ̃/ŋ/s/w/ɪ/t/h/f/ɹ/ʌ/m/d̪/ə/s/t/ɔ/ɹ/s/ɪ/k/s/s/p/ũ/n/s/ə/f/f/ɹ/ɛ/ʃ/s/n/o/ʊ/pʰ/i/s/f/a/ɪ/f/θ/ɪ/k/s/l/æ/p/s/ə/f/b/l/u/t/ʃʰ/i/z/æ̃/n/m/e/ɪ/b/i/ə/s/n/æ/k/f/ɔ/ɹ/h/b/ɹ/ʌ/ð/b/ɔ/p/w/i/ɔ/l/s/o/n/iː/d/ə/s/m/ɔ/l/p/l/æ/s/t/ɪ/k/s/n/e/ɪ/k/æ̃/n/d/ə/b/i/k/t/ɔ/ɪ/f/ɹ/ɔ/ɡ/f/ɔ/ɹ/ð/ə/kʰ/ɪ/t/s/ʃ/i/k/æ̃/n/s/k/u/p/d̪/ɪ/s/t/ɪ̃/ŋ/s/ɪ̃/n/t/u/θ/r/i/ɹ/ɛ/d/b/æ/k/s/æ̃/n/d/w/i/w/ɪ/l/ɡ/o/m/i/t/h/w/ɛ̃/n/z/d/e/ɪ/æ̝/t/ə/tʰ/ɹ/ẽ/ɪ/n/s/t/e/ɪ/ʃ/ə/n
Predicted Phones: p/l̠/i/z/k/o̝/ʊ/l̠/s̺/t̠/ɛ/l̠/ə/æ/s̺/k/h/ɜ˞/t̠/ü/p/ɹ/ɪ̃/ŋʲ/t̠/ɪ̈/s̺/t̠/ɪ̃/ŋʲ/z/wʲ/ɪ̈/ð̞/h/ɜ˞/f/ɹ/ə/m/ð̞/ə/s̺/t̠/ɔ/ɹ/s̺/ɪ̈/k/s̺/s̺/p/ũ/n/z/ə/v/f/ɹ/ɛ/ʃ/s̺/n/o̝/ʊ/p/i/z/f/a̠/ɪ̈/v/θ/ɪ̈/k/s̺/l̠/æ/p/z/ə/v/p/l̠/ü/t̠/ʃ/i/z/ə/n/d/m/e̝/ɪ̈/p/i/ə/s̺/n/æ/k/f/ɔ/ɹ/h/ɜ˞/p/ɹ/ə/ð̞/ɜ˞/p/ɑ̈/b/wʲ/i/ɔ/l̠/s̺/o̝/ʊ/n/i/d/ə/s̺/m/ɔ/l̠/p/l̠/æ/s̺/t̠/ɪ̈/k/s̺/n/e̝/ɪ̈/k/ə/n/d/ə/p/i/ɡʲ/t̠/ɔ/ɪ̈/f/ɹ/

----------------------------------------
WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/ewe1.wav
Language: eng
ASR Text: None
Ground Truth Phones: pʰ/l/iː/z/k/ɔ/sː/t̪/ɛ/l/aː/æː/s/k/h/ɛ/tʰ/ʊ/b/ɹ/ɪ̃/ŋ/d̪/iː/z/t̪/i/ə̃/z/f/ɹ/ɔ/m/d̪/ʌ/s/t/ɔ/ə/s/ɪ/k/s/s/p/<unk>/n/z/ɔ/f/fː/ɹ/ɛ/ʃ/s/n/o/ʊ/pʰ/iː/z/f/a/ɪ/f/t̪/ɪ/k/s/l/æ/b/s/ɔ/v/b/l/u/iː/z/æ̃/n/m/e/ɪ/b/i/ə/s/nː/æ/k/f/ɔ/h/ɛ/b/ɹ/ɛ/d̪/ʌ/b/ɑː/b/m̩/w/iː/ɔ/l/s/o/<unk>/n/i/d/ə/s/m/ɔ/lˠ/p/l/æ/s/i/k/s/nː/e/ɪ/k/æ/n/ə/b/ɪ/k/tʰ/<unk>/ɔ/ɪ/f/ɹ/ɔ/ɡ/f/ɔ/d̪/ʌ/k/iː/ə/d/z/ʃ/i/k/ẽ/n/s/k/u/p/d̪/i/s/θ/ɪ̃/ŋ/z/ɪ/n/t/ʊ/θ/ɹ/i/ɹ/ɛ/d/b/æː/ɡ/z/<unk>/n/w/i/w/ɪ/l/ŋ/ɡ/o/ʊ/m/i/t/h/ɛː/w/ɛ̃/z/d/e/ɪ/ɛ/d̪/ə/t/ɹ/e/ɪ̃/n/s/t/e/ɪ/ʃ/ə/n
Predicted Phones: p/l̠/i/z/k/ɑ̈/s̺/t̠/ɛ/l̠/ɜ˞/æ/s̺/k/h/ɜ˞/t̠/ü/p/ɹ/ɪ̃/ŋʲ/ð̞/i/z/θ/ɪ̃/ŋʲ/z/f/ɹ/ə/m/ð̞/ə/s̺/t̠/ɔ/ɹ/s̺/ɪ̈/k/s̺/p/ũ/n/z/ə/v/f/ɹ/ɛ/ʃ/n/o̝/ʊ/p/i/s̺/f/a̠/ɪ̈/v/θ/ɪ̈/k/s̺/l̠/æ/p/s̺/ə/v/p/l̠/ü/t̠/ʃ/i/z/ə/n/d/m/e̝/ɪ̈/p/i/ə/s̺/n/æ/k/f/ɔ/ɹ/h/ɜ˞/p/ɹ/ə/ð̞/ɜ˞/p/ɑ̈/b/wʲ/i/ɔ/l̠/s̺/o̝/ʊ/n/i/d/ə/s̺/m/ɔ/l̠/p/l̠/æ/s̺/ɪ̈/k/s̺/n/ɪ̈/

----------------------------------------
WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/tigrigna2.wav
Language: eng
ASR Text: None
Ground Truth Phones: p/l/i/s/k/ɔ/l/ə/s/t̪/e̝/lː/a/a/s/k/h/t/<unk>/b/ɹ/ɪ/ŋ/d̪/ɪ/s/t/ɪ/ŋ/ɡ/s/w/ɪ/d/h/ɜ/ɹ/f/ɹ/ɔ/m/d̪/ɛ/s/t/ɔ/ɹ/s/i/k/s/<unk>/s/p/ũ/n/s/ɔ/f/ɹ/ɛ/ʃ/s/n/o/ʔ/s/n/o/<unk>/p/i/s/f/a/ɪ/f/t̪/ɪ/k/s/l/a/b/s/ɔ/f/b/l/u/i/z/ə/n/m/e/ɪ/β/ĩ/e/s/n/a/k/f/ɔ/ɹ/h/b/ɹ/a/ʔ/b/ɹ/a/ð/ə/ɹ/b/o/b/w/i/ɔ/l/s/o/n/iː/d/e/s/m/ɔ/l/b/l/a/s/t̪/i/k/s/n/e/k/ɛ̝/n/ə/b/i/ɡ/t/ɔ/ɪ/f/r/ɔː/ɡ/f/ɔ/r/d̪/ʌ/k/iː/d/z/ʃ/i/k/ɛ̝/n/s/k/uː/p/d̪/ɪ/s/t̪/ɪ/ŋ/k/s/ɪ/n/t/u/t̪/r/iː/r/ɛ/d/b/a/ɡ/s/ɛ/n/w/i/w/<unk>/l/ɡ/o/m/i/t/x/<unk>/ɹ/w/ɛ/n/s/d/e/æ/t/d̪/ə/t/ɹ/e/ɪ̃/n/s/t/e/ɪ/ʃ/<unk>/n
Predicted Phones: p/l̠/i/z/k/ɔ/l̠/s̺/ɪ̈/l̠/ə/æ/s̺/k/t̠/h/ɜ˞/t̠/ü/p/ɹ/ɪ̃/ŋʲ/ð̞/i/z/θ/ɪ̃/ŋʲ/k/z/wʲ/ɪ̈/ð̞/h/ɜ˞/f/ɹ/ə/m/ð̞/ə/s̺/t̠/ɔ/ɹ/s̺/ɪ̈/k/s̺/s̺/p/ũ/n/z/ə/v/f/ɹ/ɛ/ʃ/s̺/n/ʊ/p/s̺/n/ʊ/p/i/z/f/a̠/ɪ̈/v/θ/ɪ̈/k/s̺/l̠/æ/p/s̺/ə/v/p/l̠/ü/t̠/ʃ/ɪ̈/z/ə/n/d/m/e̝/ɪ̈/p/i/e̝/ɪ̈/s̺/n/æ/k/f/ɹ/h/ɜ˞/p/ɹ/ə/ð̞/ɜ˞/p/ʊ/p/

----------------------------------------


In [15]:
for res in results:
    print("WAVPATH:", res["wavpath"])
    print("Language:", res["language"])
    print("ASR Text:", res["asr_text"])
    print("Ground Truth Phones:", res["phone_str"])
    print("Predicted Phones:", res["prediction"])
    print("PR Metrics:", res["pr_metrics"])
    display(play_audio(res["speech"]))
    print("-" * 40)

WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/romanian6.wav
Language: eng
ASR Text: None
Ground Truth Phones: p/l/i/z/k/ɔ/l/s/t/ɛ/l/ə/æ/s/k/h/t/u/b/r/ɪ̃/ŋ/k/d̪/i/s/t̪/ɪ̃/ŋ/s/w/ɪ/t/h/f/ɹ/ʌ/m/d̪/ə/s/t/ɔ/ɹ/s/ɪ/k/s/s/p/ũ/n/s/ə/f/f/ɹ/ɛ/ʃ/s/n/o/ʊ/pʰ/i/s/f/a/ɪ/f/θ/ɪ/k/s/l/æ/p/s/ə/f/b/l/u/t/ʃʰ/i/z/æ̃/n/m/e/ɪ/b/i/ə/s/n/æ/k/f/ɔ/ɹ/h/b/ɹ/ʌ/ð/b/ɔ/p/w/i/ɔ/l/s/o/n/iː/d/ə/s/m/ɔ/l/p/l/æ/s/t/ɪ/k/s/n/e/ɪ/k/æ̃/n/d/ə/b/i/k/t/ɔ/ɪ/f/ɹ/ɔ/ɡ/f/ɔ/ɹ/ð/ə/kʰ/ɪ/t/s/ʃ/i/k/æ̃/n/s/k/u/p/d̪/ɪ/s/t/ɪ̃/ŋ/s/ɪ̃/n/t/u/θ/r/i/ɹ/ɛ/d/b/æ/k/s/æ̃/n/d/w/i/w/ɪ/l/ɡ/o/m/i/t/h/w/ɛ̃/n/z/d/e/ɪ/æ̝/t/ə/tʰ/ɹ/ẽ/ɪ/n/s/t/e/ɪ/ʃ/ə/n
Predicted Phones: pʰ/l/i/z/kʰ/o/ʊ/l/s/t/ɛ/l/ə/æ/s/k/h/ə˞/tʰ/u/p/ɹ/ɪ̃/ŋ/t/ɪ/s/t/ɪ̃/ŋ/z/w/ɪ/ð/h/ə˞/f/ɹ/ʌ/m/ð/ə/s/t/ɔ/ɹ/s/ɪ/k/s/s/p/ũ/n/z/ʌ/v/f/ɹ/ɛ/ʃ/s/n/o/ʊ/pʰ/i/z/f/a/ɪ/v/θ/ɪ/k/s/l/æ/b/s/ʌ/v/p/l/u/t/ʃ/i/z/ə/n/d/m/e/ɪ/p/i/ə/s/n/æ/k/f/ɔ/ɹ/h/ə˞/p/ɹ/ʌ/ð/ə˞/p/ɑ/b/w/i/ɔ/l/s/o/ʊ/n/i/d/ə/s/m/ɔ/l/pʰ/l/æ/s/t/ɪ/k/s/n/e/ɪ/k/ə/n/d/ə/p/ɪ/ɡ/tʰ/ɔ/ɪ/f/ɹ/ɑ/ɡ/f/ɔ/ɹ/ð/ə/kʰ/ɪ/t/s/ʃ/i/kʰ/æ̃/n/s/k/u/p
PR Metr

----------------------------------------
WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/ewe1.wav
Language: eng
ASR Text: None
Ground Truth Phones: pʰ/l/iː/z/k/ɔ/sː/t̪/ɛ/l/aː/æː/s/k/h/ɛ/tʰ/ʊ/b/ɹ/ɪ̃/ŋ/d̪/iː/z/t̪/i/ə̃/z/f/ɹ/ɔ/m/d̪/ʌ/s/t/ɔ/ə/s/ɪ/k/s/s/p/<unk>/n/z/ɔ/f/fː/ɹ/ɛ/ʃ/s/n/o/ʊ/pʰ/iː/z/f/a/ɪ/f/t̪/ɪ/k/s/l/æ/b/s/ɔ/v/b/l/u/iː/z/æ̃/n/m/e/ɪ/b/i/ə/s/nː/æ/k/f/ɔ/h/ɛ/b/ɹ/ɛ/d̪/ʌ/b/ɑː/b/m̩/w/iː/ɔ/l/s/o/<unk>/n/i/d/ə/s/m/ɔ/lˠ/p/l/æ/s/i/k/s/nː/e/ɪ/k/æ/n/ə/b/ɪ/k/tʰ/<unk>/ɔ/ɪ/f/ɹ/ɔ/ɡ/f/ɔ/d̪/ʌ/k/iː/ə/d/z/ʃ/i/k/ẽ/n/s/k/u/p/d̪/i/s/θ/ɪ̃/ŋ/z/ɪ/n/t/ʊ/θ/ɹ/i/ɹ/ɛ/d/b/æː/ɡ/z/<unk>/n/w/i/w/ɪ/l/ŋ/ɡ/o/ʊ/m/i/t/h/ɛː/w/ɛ̃/z/d/e/ɪ/ɛ/d̪/ə/t/ɹ/e/ɪ̃/n/s/t/e/ɪ/ʃ/ə/n
Predicted Phones: pʰ/l/i/z/kʰ/ɑ/s/t/ɛ/l/ə˞/æ/s/k/h/ə˞/tʰ/u/p/ɹ/ɪ̃/ŋ/ð/i/z/θ/ɪ̃/ŋ/z/f/ɹ/ʌ/m/ð/ə/s/t/ɔ/ɹ/s/ɪ/k/s/p/ũ/n/z/ʌ/v/f/ɹ/ɛ/ʃ/n/o/ʊ/pʰ/i/s/f/a/ɪ/v/θ/ɪ/k/s/l/æ/p/s/ʌ/v/p/l/u/t/ʃ/i/z/ə/n/d/m/e/ɪ/b/i/ə/s/n/æ/k/f/ɔ/ɹ/h/ə˞/p/ɹ/ʌ/ð/ə˞/p/ɑ/b/w/i/ɔ/l/s/o/ʊ/n/i/d/ə/s/m/ɔ/l/pʰ/l/æ/s/ɪ/k/s/n/e/ɪ/k/ə/n/d/ə/p/ɪ/ɡ/tʰ/ɔ/ɪ/f/ɹ/ɔ/ɡ/f/ɔ/ɹ/

----------------------------------------
WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/tigrigna2.wav
Language: eng
ASR Text: None
Ground Truth Phones: p/l/i/s/k/ɔ/l/ə/s/t̪/e̝/lː/a/a/s/k/h/t/<unk>/b/ɹ/ɪ/ŋ/d̪/ɪ/s/t/ɪ/ŋ/ɡ/s/w/ɪ/d/h/ɜ/ɹ/f/ɹ/ɔ/m/d̪/ɛ/s/t/ɔ/ɹ/s/i/k/s/<unk>/s/p/ũ/n/s/ɔ/f/ɹ/ɛ/ʃ/s/n/o/ʔ/s/n/o/<unk>/p/i/s/f/a/ɪ/f/t̪/ɪ/k/s/l/a/b/s/ɔ/f/b/l/u/i/z/ə/n/m/e/ɪ/β/ĩ/e/s/n/a/k/f/ɔ/ɹ/h/b/ɹ/a/ʔ/b/ɹ/a/ð/ə/ɹ/b/o/b/w/i/ɔ/l/s/o/n/iː/d/e/s/m/ɔ/l/b/l/a/s/t̪/i/k/s/n/e/k/ɛ̝/n/ə/b/i/ɡ/t/ɔ/ɪ/f/r/ɔː/ɡ/f/ɔ/r/d̪/ʌ/k/iː/d/z/ʃ/i/k/ɛ̝/n/s/k/uː/p/d̪/ɪ/s/t̪/ɪ/ŋ/k/s/ɪ/n/t/u/t̪/r/iː/r/ɛ/d/b/a/ɡ/s/ɛ/n/w/i/w/<unk>/l/ɡ/o/m/i/t/x/<unk>/ɹ/w/ɛ/n/s/d/e/æ/t/d̪/ə/t/ɹ/e/ɪ̃/n/s/t/e/ɪ/ʃ/<unk>/n
Predicted Phones: pʰ/l/i/z/kʰ/ɔ/l/s/ɪ/l/ə/æ/s/k/t/h/ə˞/tʰ/u/p/ɹ/ɪ̃/ŋ/ð/i/z/θ/ɪ̃/ŋ/z/w/ɪ/ð/h/ə˞/f/ɹ/ʌ/m/ð/ə/s/t/ɔ/ɹ/s/ɪ/k/s/s/p/ũ/n/z/ʌ/v/f/ɹ/ɛ/ʃ/s/n/ʊ/p/s/n/o/ʊ/p/i/z/f/a/ɪ/v/θ/ɪ/k/s/l/æ/p/s/ʌ/v/p/l/u/t/ʃ/i/z/ə/n/d/m/e/ɪ/b/i/ɪ/s/n/æ/k/f/ɔ/ɹ/h/ə˞/p/ɹ/ʌ/ð/ə˞/p/ɑ/b/w/i/ɔ/l/s/o/ʊ/n/i/d/ə/s/m/ɔ/l/pʰ/l/æ/

----------------------------------------
